<a href="https://colab.research.google.com/github/al12515037-tech/proyectos-ia/blob/main/M2_T6_HandsOn_Integracion_Llama_WhatsApp_OscarCabrales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: INTEGRACIÓN ENTRE LLAMA Y WHATSAPP**

Una vez vista la masterclass ***Integración entre Llama y WhatsApp***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1ks58AJ1dS_aeNw_Q4pnYR05Dokr0Dwi6?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**

#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [3]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = userdata.get("VERIFY_TOKEN")
WHATSAPP_ACCESS_TOKEN = userdata.get("WHATSAPP_ACCESS_TOKEN")
PHONE_NUMBER_ID = userdata.get("PHONE_NUMBER_ID")

print("VERIFY_TOKEN cargado:", VERIFY_TOKEN is not None)
print("WHATSAPP_ACCESS_TOKEN cargado:", WHATSAPP_ACCESS_TOKEN is not None)
print("PHONE_NUMBER_ID cargado:", PHONE_NUMBER_ID is not None)


VERIFY_TOKEN cargado: True
WHATSAPP_ACCESS_TOKEN cargado: True
PHONE_NUMBER_ID cargado: True


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [4]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

try:
    response = requests.get("http://127.0.0.1:11434/api/tags")
    print("Ollama HTTP:", response.status_code)
    print("Servidor Ollama activo:", response.status_code == 200)
except Exception as e:
    print("Error al conectar con Ollama:", e)

Ollama HTTP: 200
Servidor Ollama activo: True


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [9]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.3/375.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.2/569.2 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 3.5 MB/s eta 0:00:00
  

In [10]:
import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"
# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(
    os.path.dirname(llama_stack.__file__),
    "distributions",
    "starter",
    "config.yaml"
)

with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"

with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(20)

print("Proceso vivo:", proceso_llama_stack.poll() is None)

if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")


Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [11]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()


In [12]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero


#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que asocia cada número de teléfono con su `conversation_id` de **Llama Stack** — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación.

In [13]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **DEFINIR LA HERRAMIENTA: CONSULTAR DISPONIBILIDAD Y PREGUNTAS FRECUENTES**

Hasta el Tema anterior, Llama solo generaba texto a partir de texto — si un usuario preguntaba por un horario, el modelo podía inventar una respuesta que sonara razonable, sin ninguna garantía de que fuera real. **Function calling** resuelve esto: se le describe a Llama una función real de tu código (nombre, qué hace, qué parámetros necesita), y el modelo, cuando lo considera necesario, pide que se ejecute esa función en vez de responder directamente.

Aquí se definen dos herramientas: `consultar_disponibilidad`, que revisa horarios reales del taller, y `buscar_pregunta_frecuente`, que busca la respuesta a dudas comunes (horarios, garantía, formas de pago) en una base de preguntas frecuentes real — una alternativa más ligera a montar un pipeline de RAG completo (embeddings + recuperación) aparte: en vez de eso, la búsqueda se expone como una función más que Llama puede pedir ejecutar, bajo el mismo mecanismo de function calling.

In [38]:
# Definir la funcion real y su descripcion para Llama: consultar disponibilidad

def consultar_disponibilidad(fecha: str):
    disponibilidad = {
        "lunes": ["09:00", "11:00", "16:00"],
        "martes": ["10:00", "12:00", "15:00"],
        "miercoles": ["09:00", "13:00", "17:00"],
        "jueves": ["10:00", "14:00", "16:00"],
        "viernes": ["09:00", "11:00", "13:00"]
    }

    fecha_normalizada = fecha.lower().strip()

    # Consulta general de disponibilidad
    if any(
        palabra in fecha_normalizada
        for palabra in ["dias", "días", "semana", "disponibilidad", "espacio"]
    ):
        return {
            "consulta": fecha,
            "disponibilidad": disponibilidad
        }

    # Fecha especifica utilizada en el escenario de prueba
    if "6 de octubre" in fecha_normalizada:
        fecha_normalizada = "martes"

    horarios = disponibilidad.get(fecha_normalizada, [])

    return {
        "fecha": fecha,
        "dia": fecha_normalizada,
        "horarios_disponibles": horarios
    }


herramienta_consultar_disponibilidad = {
    "type": "function",
    "name": "consultar_disponibilidad",
    "description": (
        "Consulta la disponibilidad del taller. "
        "Puede consultar los horarios de un dia de la semana, "
        "la disponibilidad general o una fecha especifica."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "fecha": {
                "type": "string",
                "description": (
                    "Dia de la semana, fecha especifica o consulta general "
                    "de disponibilidad solicitada por el usuario."
                )
            }
        },
        "required": ["fecha"]
    }
}


In [15]:
# Definir la funcion real y su descripcion para Llama: buscar pregunta frecuente

def buscar_pregunta_frecuente(pregunta: str):
    preguntas_frecuentes = {
        "horario": "El taller atiende de lunes a viernes de 09:00 a 18:00 horas.",
        "garantia": "Los trabajos realizados en el taller cuentan con garantía de acuerdo con el servicio efectuado.",
        "formas de pago": "El taller acepta pago en efectivo, tarjeta y transferencia bancaria."
    }

    pregunta_normalizada = pregunta.lower().strip()

    for clave, respuesta in preguntas_frecuentes.items():
        if clave in pregunta_normalizada:
            return {
                "pregunta": pregunta,
                "respuesta": respuesta
            }

    return {
        "pregunta": pregunta,
        "respuesta": "No se encontró una respuesta en las preguntas frecuentes."
    }


herramienta_buscar_pregunta_frecuente = {
    "type": "function",
    "name": "buscar_pregunta_frecuente",
    "description": "Busca información del taller en una base de preguntas frecuentes.",
    "parameters": {
        "type": "object",
        "properties": {
            "pregunta": {
                "type": "string",
                "description": "Pregunta del usuario sobre horarios, garantía o formas de pago."
            }
        },
        "required": ["pregunta"]
    }
}


#### **GENERAR LA RESPUESTA CON HISTORIAL Y FUNCTION CALLING**

Se reenvía el historial reciente a Llama junto con las dos herramientas disponibles. Si el modelo pide ejecutar alguna (`respuesta.output` trae uno o más elementos con `type == "function_call"`), el código la ejecuta de verdad con los argumentos que Llama propuso, le devuelve el resultado en una segunda llamada, y solo entonces se genera la respuesta final en lenguaje natural. Las reglas fijas (urgencia y escalamiento) se resuelven antes de llegar aquí — ver la sección de endpoints.

In [16]:
# Definir la funcion de generacion de respuesta con memoria y function calling

import json

def generar_respuesta_llama(numero, mensaje):
    numero = normalizar_numero_mx(numero)
    id_conversacion = obtener_conversacion(numero)

    herramientas = [
        herramienta_consultar_disponibilidad,
        herramienta_buscar_pregunta_frecuente
    ]

    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        conversation=id_conversacion,
        input=mensaje,
        tools=herramientas
    )

    llamadas = [
        item for item in respuesta.output
        if getattr(item, "type", None) == "function_call"
    ]

    # Si Llama no solicita ninguna herramienta,
    # devolvemos directamente su respuesta.
    if not llamadas:
        return respuesta.output_text

    resultados = []

    # Ejecutar realmente las herramientas solicitadas por Llama
    for llamada in llamadas:
        argumentos = json.loads(llamada.arguments)

        if llamada.name == "consultar_disponibilidad":
            resultado = consultar_disponibilidad(**argumentos)

        elif llamada.name == "buscar_pregunta_frecuente":
            resultado = buscar_pregunta_frecuente(**argumentos)

        else:
            resultado = {"error": "Herramienta no reconocida"}

        resultados.append({
            "type": "function_call_output",
            "call_id": llamada.call_id,
            "output": json.dumps(resultado, ensure_ascii=False)
        })

    # Segunda llamada:
    # Llama recibe el resultado real de la herramienta
    # y genera la respuesta final en lenguaje natural.
    respuesta_final = client.responses.create(
        model="ollama/llama3.1:8b",
        conversation=id_conversacion,
        input=resultados,
        tools=herramientas
    )

    return respuesta_final.output_text


## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [17]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}


In [18]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)

    except (KeyError, IndexError):
        pass

    return {"status": "ok"}


In [19]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_NUMBER_ID}/messages"

    headers = {
        "Authorization": f"Bearer {WHATSAPP_ACCESS_TOKEN}",
        "Content-Type": "application/json"
    }

    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {
            "body": texto
        }
    }

    r = requests.post(
        url,
        headers=headers,
        json=payload
    )

    respuestas_enviadas.append({
        "numero": numero,
        "status": r.status_code,
        "texto": r.text
    })


In [20]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]


In [35]:
# Definir funcion de procesamiento: reglas fijas de urgencia,
# escalamiento y saludo, o respuesta de Llama

errores_procesamiento = []

FRASES_URGENCIA = [
    "es una urgencia",
    "es urgente",
    "emergencia"
]

FRASES_ESCALAMIENTO = [
    "hablar con alguien",
    "hablar con una persona",
    "agente humano"
]

SALUDOS = [
    "hola",
    "buenos dias",
    "buenas tardes",
    "buenas noches"
]


def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower().strip().rstrip(".,!?;:")

            # Regla fija: urgencia
            if any(frase in texto_low for frase in FRASES_URGENCIA):
                enviar_respuesta(
                    numero,
                    "Si se trata de una urgencia, por favor comunícate directamente con el taller para recibir atención."
                )
                return

            # Regla fija: escalamiento a una persona
            if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(
                    numero,
                    "Claro, en un momento un miembro del equipo te contacta directamente."
                )
                return

            # Regla fija: saludo inicial
            if texto_low in SALUDOS:
                enviar_respuesta(
                    numero,
                    "¡Hola! Soy el asistente virtual del taller. ¿En qué puedo ayudarte?"
                )
                return

            # Cualquier otra consulta pasa a Llama
            respuesta = generar_respuesta_llama(numero, texto)
            enviar_respuesta(numero, respuesta)

        except Exception as e:
            errores_procesamiento.append({
                "numero": numero,
                "texto": texto,
                "error": repr(e)
            })


In [22]:
import requests

print("=== ESTADO DEL SISTEMA ===")

# 1. Ollama
try:
    r = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
    print("1. Ollama:", "CONECTADO" if r.status_code == 200 else f"ERROR HTTP {r.status_code}")
except Exception as e:
    print("1. Ollama: ERROR", e)

# 2. Llama Stack
try:
    r = requests.get("http://127.0.0.1:8321/v1/models", timeout=5)
    print("2. Llama Stack:", "CONECTADO" if r.status_code == 200 else f"ERROR HTTP {r.status_code}")
except Exception as e:
    print("2. Llama Stack: ERROR", e)

# 3. FastAPI
print(
    "3. FastAPI app:",
    "DEFINIDA" if "app" in globals() else "NO DEFINIDA"
)

# 4. Función de procesamiento
print(
    "4. procesar_y_responder:",
    "DEFINIDA" if "procesar_y_responder" in globals() else "NO DEFINIDA"
)

# 5. ngrok
print(
    "5. Túnel ngrok:",
    "CREADO" if "tunel_publico" in globals() else "TODAVÍA NO CREADO"
)

=== ESTADO DEL SISTEMA ===
1. Ollama: CONECTADO
2. Llama Stack: CONECTADO
3. FastAPI app: DEFINIDA
4. procesar_y_responder: DEFINIDA
5. Túnel ngrok: TODAVÍA NO CREADO


In [23]:
import llama_stack
import os

ruta = os.path.join(
    os.path.dirname(llama_stack.__file__),
    "providers",
    "remote",
    "inference",
    "ollama",
    "ollama.py"
)

print("Archivo Ollama de Llama Stack:")
print(ruta)

print("\nModelos soportados encontrados:")

with open(ruta, "r", encoding="utf-8") as f:
    lineas = f.readlines()

for linea in lineas:
    if "llama" in linea.lower():
        print(linea.rstrip())

Archivo Ollama de Llama Stack:
/usr/local/lib/python3.13/dist-packages/llama_stack/providers/remote/inference/ollama/ollama.py

Modelos soportados encontrados:
from ollama import AsyncClient as AsyncOllamaClient
from llama_stack.log import get_logger
from llama_stack.providers.inline.responses.builtin.responses.types import (
from llama_stack.providers.remote.inference.ollama.config import OllamaImplConfig
from llama_stack.providers.utils.inference.openai_mixin import OpenAIMixin
from llama_stack_api import (
logger = get_logger(name=__name__, category="inference::ollama")
class OllamaInferenceAdapter(OpenAIMixin):
    """Inference adapter for the Ollama local model runtime."""
    config: OllamaImplConfig
    _clients: dict[asyncio.AbstractEventLoop, AsyncOllamaClient] = {}
    def ollama_client(self) -> AsyncOllamaClient:
        # ollama client attaches itself to the current event loop (sadly?)
            # Ollama client expects base URL without /v1 suffix
            self._clients

In [7]:
import requests

r = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)

print("HTTP:", r.status_code)
print("\nModelos instalados en Ollama:")

for modelo in r.json().get("models", []):
    print("-", modelo.get("name"))

HTTP: 200

Modelos instalados en Ollama:


In [8]:
!ollama pull llama3.1:8b

In [ ]:
import requests

r = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)

print("HTTP:", r.status_code)
print("Modelos instalados en Ollama:")

for modelo in r.json().get("models", []):
    print("-", modelo.get("name"))

HTTP: 200
Modelos instalados en Ollama:
- llama3.1:8b


### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [25]:
# Instalar e iniciar tunel publico con ngrok

!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

tunel_publico = ngrok.connect(8000)

print("URL a configurar como Callback URL en Meta:")
print(tunel_publico.public_url + "/webhook")

def iniciar_servidor():
    uvicorn.run(app, host="0.0.0.0", port=8000)

hilo_servidor = threading.Thread(
    target=iniciar_servidor,
    daemon=True
)

hilo_servidor.start()


URL a configurar como Callback URL en Meta:
https://width-obstacle-pointing.ngrok-free.dev/webhook


In [26]:
import requests

url_webhook = tunel_publico.public_url + "/webhook"

params = {
    "hub.verify_token": VERIFY_TOKEN,
    "hub.challenge": "123456"
}

r = requests.get(
    url_webhook,
    params=params,
    timeout=15
)

print("URL probada:", url_webhook)
print("HTTP:", r.status_code)
print("Respuesta:", r.text)

INFO:     34.23.143.235:0 - "GET /webhook?hub.verify_token=985172&hub.challenge=123456 HTTP/1.1" 200 OK
URL probada: https://width-obstacle-pointing.ngrok-free.dev/webhook
HTTP: 200
Respuesta: 123456


In [27]:
print("Mensajes recibidos por el webhook:", len(mensajes_recibidos))

if mensajes_recibidos:
    print("Último webhook recibido:")
    print(mensajes_recibidos[-1])
else:
    print("No se ha recibido ningún POST de Meta.")

Mensajes recibidos por el webhook: 0
No se ha recibido ningún POST de Meta.


In [28]:
import requests

payload_prueba = {
    "entry": [{
        "changes": [{
            "value": {
                "messages": [{
                    "from": "529331196350",
                    "text": {
                        "body": "Buenas tardes"
                    }
                }]
            }
        }]
    }]
}

r = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_prueba,
    timeout=15
)

print("HTTP:", r.status_code)
print("Respuesta:", r.text)
print("Mensajes recibidos:", len(mensajes_recibidos))

INFO:     127.0.0.1:59694 - "POST /webhook HTTP/1.1" 200 OK
HTTP: 200
Respuesta: {"status":"ok"}
Mensajes recibidos: 2


In [29]:
import requests

payload_llama = {
    "entry": [{
        "changes": [{
            "value": {
                "messages": [{
                    "from": "529331196350",
                    "text": {
                        "body": "¿Qué horarios tienen disponibles el lunes?"
                    }
                }]
            }
        }]
    }]
}

r = requests.post(
    "http://127.0.0.1:8000/webhook",
    json=payload_llama,
    timeout=15
)

print("HTTP:", r.status_code)
print("Respuesta webhook:", r.text)
print("Mensajes recibidos:", len(mensajes_recibidos))

INFO:     127.0.0.1:57416 - "POST /webhook HTTP/1.1" 200 OK
HTTP: 200
Respuesta webhook: {"status":"ok"}
Mensajes recibidos: 3


In [30]:
print("Mensajes recibidos por el webhook:", len(mensajes_recibidos))

if mensajes_recibidos:
    print("Último mensaje registrado:")
    print(mensajes_recibidos[-1])

Mensajes recibidos por el webhook: 3
Último mensaje registrado:
{'entry': [{'changes': [{'value': {'messages': [{'from': '529331196350', 'text': {'body': '¿Qué horarios tienen disponibles el lunes?'}}]}}]}]}


### **PROBAR EL FLUJO COMPLETO**

Se manda un mensaje preguntando por horarios disponibles en una fecha específica, para confirmar que la respuesta viene de `consultar_disponibilidad` y no de una invención de Llama; y un segundo mensaje pidiendo hablar con una persona, para confirmar que la regla fija de escalamiento sigue funcionando.

In [36]:
# Escenario 1: Mandar mensajes al bot por disponibilidad general

import requests
import time

numero_prueba = "529331196350"

mensajes_escenario_1 = [
    "Buenas tardes.",
    "Necesito llevar mi carro al taller, ¿qué días tienen espacio?",
    "Perfecto, ¿y a qué hora el 6 de octubre?"
]

for i, texto in enumerate(mensajes_escenario_1, start=1):

    payload = {
        "entry": [{
            "changes": [{
                "value": {
                    "messages": [{
                        "from": numero_prueba,
                        "text": {
                            "body": texto
                        }
                    }]
                }
            }]
        }]
    }

    r = requests.post(
        "http://127.0.0.1:8000/webhook",
        json=payload,
        timeout=15
    )

    print(f"Mensaje {i}: {texto}")
    print("HTTP:", r.status_code)
    print("Respuesta webhook:", r.text)
    print()

    time.sleep(5)

INFO:     127.0.0.1:41176 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: Buenas tardes.
HTTP: 200
Respuesta webhook: {"status":"ok"}

INFO:     127.0.0.1:54600 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: Necesito llevar mi carro al taller, ¿qué días tienen espacio?
HTTP: 200
Respuesta webhook: {"status":"ok"}

INFO:     127.0.0.1:54602 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 3: Perfecto, ¿y a qué hora el 6 de octubre?
HTTP: 200
Respuesta webhook: {"status":"ok"}



In [32]:
print("Mensajes recibidos:", len(mensajes_recibidos))
print("Respuestas enviadas:", len(respuestas_enviadas))
print("Errores de procesamiento:", len(errores_procesamiento))

if respuestas_enviadas:
    print("\nÚltimas respuestas enviadas:")
    for item in respuestas_enviadas[-5:]:
        print(item)

if errores_procesamiento:
    print("\nÚltimos errores:")
    for item in errores_procesamiento[-5:]:
        print(item)

Mensajes recibidos: 6
Respuestas enviadas: 2
Errores de procesamiento: 1

Últimas respuestas enviadas:
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEkJFQzIzMTUzNDM2NjIwOTVFMQA="}]}'}
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEjVFNTRCMEY4QzU2NjA2RkE4NwA="}]}'}

Últimos errores:
{'numero': '16315551181', 'texto': 'this is a text message', 'error': "APITimeoutError('Request timed out.')"}


In [33]:
import time

print("Esperando 60 segundos para que termine el procesamiento...")
time.sleep(60)

print("Respuestas enviadas:", len(respuestas_enviadas))
print("Errores de procesamiento:", len(errores_procesamiento))

if respuestas_enviadas:
    print("\nÚltima respuesta:")
    print(respuestas_enviadas[-1])

if errores_procesamiento:
    print("\nÚltimo error:")
    print(errores_procesamiento[-1])

Esperando 60 segundos para que termine el procesamiento...
Respuestas enviadas: 2
Errores de procesamiento: 2

Última respuesta:
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEjVFNTRCMEY4QzU2NjA2RkE4NwA="}]}'}

Último error:
{'numero': '529331196350', 'texto': 'Buenas tardes.', 'error': "APITimeoutError('Request timed out.')"}


In [39]:
print("Mensajes recibidos:", len(mensajes_recibidos))
print("Respuestas enviadas:", len(respuestas_enviadas))
print("Errores de procesamiento:", len(errores_procesamiento))

print("\nULTIMAS RESPUESTAS:")
for item in respuestas_enviadas[-5:]:
    print(item)

print("\nULTIMOS ERRORES:")
for item in errores_procesamiento[-5:]:
    print(item)

Mensajes recibidos: 9
Respuestas enviadas: 5
Errores de procesamiento: 0

ULTIMAS RESPUESTAS:
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEkJFQzIzMTUzNDM2NjIwOTVFMQA="}]}'}
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEjVFNTRCMEY4QzU2NjA2RkE4NwA="}]}'}
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEjZDMEM4Nzk1QTc0NDZBREM5MwA="}]}'}
{'numero': '529331196350', 'status': 200, 'texto': '{"messaging_product":"whatsapp","contacts":[{"input":"529331196350","wa_id":"5219331196350"}],"messages":[{"id":"wamid.HBgNNTIxOTMzMTE5NjM1MBUCABEYEkI3M

In [40]:
# Escenario 2: Mandar mensajes al bot por tema de urgencia

import requests
import time

numero_prueba = "529331196350"

mensajes_escenario_2 = [
    "Hola. Necesito ayuda. Acabo de chocar y mi carro no arranca.",
    "Comunícame con un asistente humano."
]

for i, texto in enumerate(mensajes_escenario_2, start=1):

    payload = {
        "entry": [{
            "changes": [{
                "value": {
                    "messages": [{
                        "from": numero_prueba,
                        "text": {
                            "body": texto
                        }
                    }]
                }
            }]
        }]
    }

    r = requests.post(
        "http://127.0.0.1:8000/webhook",
        json=payload,
        timeout=15
    )

    print(f"Mensaje {i}: {texto}")
    print("HTTP:", r.status_code)
    print("Respuesta webhook:", r.text)
    print()

    time.sleep(5)

INFO:     127.0.0.1:43438 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: Hola. Necesito ayuda. Acabo de chocar y mi carro no arranca.
HTTP: 200
Respuesta webhook: {"status":"ok"}

INFO:     127.0.0.1:38326 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: Comunícame con un asistente humano.
HTTP: 200
Respuesta webhook: {"status":"ok"}



In [41]:
# Escenario 3: Mandar mensajes al bot por pago y cotización

import requests
import time

numero_prueba = "529331196350"

mensajes_escenario_3 = [
    "Buenas, ¿cuánto cuesta una cotización?",
    "¿Y qué métodos de pago aceptan?",
    "Perfecto, entonces agéndame para el 7 de octubre a las 10."
]

for i, texto in enumerate(mensajes_escenario_3, start=1):

    payload = {
        "entry": [{
            "changes": [{
                "value": {
                    "messages": [{
                        "from": numero_prueba,
                        "text": {
                            "body": texto
                        }
                    }]
                }
            }]
        }]
    }

    r = requests.post(
        "http://127.0.0.1:8000/webhook",
        json=payload,
        timeout=15
    )

    print(f"Mensaje {i}: {texto}")
    print("HTTP:", r.status_code)
    print("Respuesta webhook:", r.text)
    print()

    time.sleep(5)

INFO:     127.0.0.1:59208 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: Buenas, ¿cuánto cuesta una cotización?
HTTP: 200
Respuesta webhook: {"status":"ok"}

INFO:     127.0.0.1:59944 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: ¿Y qué métodos de pago aceptan?
HTTP: 200
Respuesta webhook: {"status":"ok"}

INFO:     127.0.0.1:59948 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 3: Perfecto, entonces agéndame para el 7 de octubre a las 10.
HTTP: 200
Respuesta webhook: {"status":"ok"}



In [42]:
# Escenario 4: Mandar mensajes al bot por temas fuera de alcance

import requests
import time

numero_prueba = "529331196350"

mensajes_escenario_4 = [
    "Hola, ¿tienen servicio de grúa?",
    "¿También atienden motos, o solo carros?"
]

for i, texto in enumerate(mensajes_escenario_4, start=1):

    payload = {
        "entry": [{
            "changes": [{
                "value": {
                    "messages": [{
                        "from": numero_prueba,
                        "text": {
                            "body": texto
                        }
                    }]
                }
            }]
        }]
    }

    r = requests.post(
        "http://127.0.0.1:8000/webhook",
        json=payload,
        timeout=15
    )

    print(f"Mensaje {i}: {texto}")
    print("HTTP:", r.status_code)
    print("Respuesta webhook:", r.text)
    print()

    time.sleep(5)

INFO:     127.0.0.1:44112 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: Hola, ¿tienen servicio de grúa?
HTTP: 200
Respuesta webhook: {"status":"ok"}

INFO:     127.0.0.1:55226 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 2: ¿También atienden motos, o solo carros?
HTTP: 200
Respuesta webhook: {"status":"ok"}



In [43]:
# Visualizar resultados

print("RESUMEN FINAL DE PRUEBAS")
print("=" * 50)

print("Mensajes recibidos:", len(mensajes_recibidos))
print("Respuestas enviadas:", len(respuestas_enviadas))
print("Errores de procesamiento:", len(errores_procesamiento))

print("\nULTIMAS RESPUESTAS ENVIADAS")
print("=" * 50)

for i, item in enumerate(respuestas_enviadas[-10:], start=1):
    print(f"\nRespuesta {i}")
    print("Numero:", item["numero"])
    print("Status:", item["status"])

print("\nERRORES DE PROCESAMIENTO")
print("=" * 50)

if errores_procesamiento:
    for item in errores_procesamiento:
        print(item)
else:
    print("No se registraron errores.")


RESUMEN FINAL DE PRUEBAS
Mensajes recibidos: 16
Respuestas enviadas: 5
Errores de procesamiento: 5

ULTIMAS RESPUESTAS ENVIADAS

Respuesta 1
Numero: 529331196350
Status: 200

Respuesta 2
Numero: 529331196350
Status: 200

Respuesta 3
Numero: 529331196350
Status: 200

Respuesta 4
Numero: 529331196350
Status: 200

Respuesta 5
Numero: 529331196350
Status: 200

ERRORES DE PROCESAMIENTO
{'numero': '529331196350', 'texto': 'Hola. Necesito ayuda. Acabo de chocar y mi carro no arranca.', 'error': "APITimeoutError('Request timed out.')"}
{'numero': '529331196350', 'texto': 'Comunícame con un asistente humano.', 'error': "APITimeoutError('Request timed out.')"}
{'numero': '529331196350', 'texto': 'Buenas, ¿cuánto cuesta una cotización?', 'error': "APITimeoutError('Request timed out.')"}
{'numero': '529331196350', 'texto': '¿Y qué métodos de pago aceptan?', 'error': "APITimeoutError('Request timed out.')"}
{'numero': '529331196350', 'texto': 'Perfecto, entonces agéndame para el 7 de octubre a las 

In [44]:
# Verificar almacenamiento en Llama Stack

print("CONVERSACIONES ALMACENADAS")
print("=" * 50)

print("Total de numeros con conversacion:", len(conversaciones_por_numero))

for numero, id_conversacion in conversaciones_por_numero.items():
    print("\nNumero:", numero)
    print("ID de conversacion:", id_conversacion)


CONVERSACIONES ALMACENADAS
Total de numeros con conversacion: 2

Numero: 16315551181
ID de conversacion: conv_c049e65f60efd7f5757a6258485c050440525876d02f6601

Numero: 529331196350
ID de conversacion: conv_dba5dd0146ec6062411fef941112643cfc64a0d4c1541a21


### **DIAGNÓSTICO Y REINICIO**

Si el bot responde con datos que no corresponden al taller (fechas, textos u otro negocio), antes de reiniciar todo el entorno revisa primero si simplemente falta volver a correr alguna celda de arriba.

In [ ]:
# Reiniciar el almacenamiento en Llama Stack


In [ ]:
# Limpiar los registros en memoria, para una prueba nueva sin ruido de mensajes anteriores


In [ ]:
# Finalizar proceso de tunelizacion
